In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("Train Size:", len(train_dataset))
print("Test Size :", len(test_dataset))

100%|██████████| 170M/170M [22:22<00:00, 127kB/s]    


Train Size: 50000
Test Size : 10000


In [2]:
class PatchEmbedding(nn.Module):

    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_channels=3,
        embed_dim=128
    ):
        super().__init__()

        self.patch_size = patch_size

        self.num_patches = (
            img_size // patch_size
        ) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):

        x = self.proj(x) # [B,128,8,8]
        x = x.flatten(2)  # [B,128,64]
        x = x.transpose(1,2)  # [B,64,128]

        return x

import torch
import torch.nn as nn

class PositionalEmbedding(nn.Module):

    def __init__(
        self,
        num_patches=64,
        embed_dim=128
    ):
        super().__init__()

        self.pos_embed = nn.Parameter( # scaling
            torch.randn(
                1,
                num_patches,
                embed_dim
            ) * 0.02
        )

    def forward(self, x):

        return x + self.pos_embed

class SimpleViTEncoder(nn.Module): #learning relationship

    def __init__(
        self,
        embed_dim=128,
        num_heads=4,
        depth=4
    ):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth
        )

    def forward(self, x):

        return self.transformer(x)


class MaskedImageEncoder(nn.Module):

    def __init__(
        self,
        img_size=32,
        patch_size=4,
        embed_dim=128,
        num_heads=4,
        depth=4
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=embed_dim
        )

        self.pos_embed = PositionalEmbedding(
            num_patches=self.patch_embed.num_patches,
            embed_dim=embed_dim
        )

        self.encoder = SimpleViTEncoder(
            embed_dim=embed_dim,
            num_heads=num_heads,
            depth=depth
        )

    def forward(self, images, mask=None):
        x = self.patch_embed(images)   # [B, 64, 128]
        x = self.pos_embed(x)          # [B, 64, 128]

        if mask is not None:
            mask = mask.to(x.device)

            B, N, D = x.shape

            x = x[mask].view(B, -1, D) # [B, num_visible, 128]

        x = self.encoder(x)

        return x

In [3]:
def generate_block_masks( #masking
    batch_size,
    grid_size=8,
    target_block_size=4
):
    num_patches = grid_size * grid_size

    context_mask = torch.ones( batch_size, num_patches, dtype=torch.bool)

    target_mask = torch.zeros( batch_size, num_patches, dtype=torch.bool)

    for b in range(batch_size):
        max_start = grid_size - target_block_size

        row_start = torch.randint(0, max_start + 1, (1,)).item()
        col_start = torch.randint(0, max_start + 1, (1,)).item()

        for r in range(row_start, row_start + target_block_size):
            for c in range(col_start, col_start + target_block_size):
                patch_index = r * grid_size + c

                context_mask[b, patch_index] = False
                target_mask[b, patch_index] = True

    return context_mask, target_mask


class SimplePredictor(nn.Module):

    def __init__(
        self,
        num_patches=64,
        embed_dim=128,
        num_heads=4,
        depth=2
    ):
        super().__init__()

        self.pos_embed = nn.Parameter(
            torch.randn(1, num_patches, embed_dim) * 0.02
        )

        self.mask_token = nn.Parameter(
            torch.randn(1, 1, embed_dim) * 0.02
        )

        predictor_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True
        )

        self.predictor = nn.TransformerEncoder(
            predictor_layer,
            num_layers=depth
        )

    def forward(
        self,
        context_reps,
        target_mask
    ):
        # context_reps : [B, num_context, 128]
        # target_mask  : [B, 64]
        # returns      : [B, num_targets, 128]

        B, num_context, D = context_reps.shape

        all_predictions = []

        for b in range(B):
            context_tokens = context_reps[b]

            target_pos = self.pos_embed[
                0,
                target_mask[b]
            ]

            num_targets = target_pos.shape[0]

            mask_tokens = self.mask_token.squeeze(0).expand(
                num_targets,
                D
            )

            target_tokens = mask_tokens + target_pos

            predictor_input = torch.cat(
                [context_tokens, target_tokens],
                dim=0
            )

            predictor_input = predictor_input.unsqueeze(0)

            output = self.predictor(predictor_input)

            target_predictions = output[
                :,
                -num_targets:,
                :
            ]

            all_predictions.append(
                target_predictions.squeeze(0)
            )

        return torch.stack(all_predictions, dim=0)

import copy
import torch.nn.functional as F

class IJEPA(nn.Module):

    def __init__(
        self,
        img_size=32,
        patch_size=4,
        embed_dim=128,
        num_heads=4,
        encoder_depth=4,
        predictor_depth=2
    ):
        super().__init__()

        self.context_encoder = MaskedImageEncoder(
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
            num_heads=num_heads,
            depth=encoder_depth
        )

        self.target_encoder = copy.deepcopy(self.context_encoder)

        for param in self.target_encoder.parameters(): #Freeze
            param.requires_grad = False

        self.predictor = SimplePredictor(
            num_patches=self.context_encoder.patch_embed.num_patches,
            embed_dim=embed_dim,
            num_heads=num_heads,
            depth=predictor_depth
        )

    def forward(self, images, context_mask, target_mask):

        context_mask = context_mask.to(images.device)
        target_mask = target_mask.to(images.device)

        context_reps = self.context_encoder(
            images,
            mask=context_mask
        )

        predicted_reps = self.predictor(
            context_reps,
            target_mask
        )

        with torch.no_grad():
            target_reps_all = self.target_encoder(images)

            B, N, D = target_reps_all.shape

            target_reps = target_reps_all[target_mask].view(B, -1, D)

        loss = F.mse_loss( predicted_reps, target_reps)

        return loss, predicted_reps, target_reps


def update_target_encoder(
    context_encoder,
    target_encoder,
    momentum=0.99
):
    for context_param, target_param in zip(
        context_encoder.parameters(),
        target_encoder.parameters()
    ):
        target_param.data = (
            momentum * target_param.data
            + (1.0 - momentum) * context_param.data
        )

In [4]:
class LinearProbe(nn.Module):

    def __init__(
        self,
        backbone,
        embed_dim=128,
        num_classes=10
    ):
        super().__init__()

        self.backbone = backbone

        for param in self.backbone.parameters():
            param.requires_grad = False

        self.classifier = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, images):

        with torch.no_grad():
            reps = self.backbone(images)   # [B, 64, 128]

        features = reps.mean(dim=1)        # [B, 128]

        logits = self.classifier(features) # [B, 10]

        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MyModel() 

if torch.cuda.device_count() > 1:
    print(f"Awesome, let's use {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model.to(device)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

model = IJEPA().to(device)

optimizer = torch.optim.AdamW(
    list(model.context_encoder.parameters())
    + list(model.predictor.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    total_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)

        context_mask, target_mask = generate_block_masks(
            batch_size=images.shape[0],
            grid_size=8,
            target_block_size=4
        )

        loss, predicted_reps, target_reps = model(
            images,
            context_mask,
            target_mask
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        update_target_encoder(
            model.context_encoder,
            model.target_encoder,
            momentum=0.99
        )

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"I-JEPA Loss: {avg_loss:.4f}"
    )


torch.save(
    model.context_encoder.state_dict(),
    "ijepa_context_encoder.pth"
)


linear_probe = LinearProbe(
    backbone=model.context_encoder
).to(device)

probe_optimizer = torch.optim.AdamW(
    linear_probe.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

probe_epochs = 10

for epoch in range(probe_epochs):

    linear_probe.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        logits = linear_probe(images)

        loss = criterion(logits, labels)

        probe_optimizer.zero_grad()
        loss.backward()
        probe_optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = 100.0 * correct / total

    linear_probe.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = linear_probe(images)
            preds = logits.argmax(dim=1)

            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_accuracy = 100.0 * test_correct / test_total

    print(
        f"Linear Probe Epoch [{epoch + 1}/{probe_epochs}] "
        f"Loss: {avg_loss:.4f} "
        f"Train Accuracy: {accuracy:.2f}% "
        f"Test Accuracy: {test_accuracy:.2f}%"
    )

Epoch [1/10] I-JEPA Loss: 0.4385
Epoch [2/10] I-JEPA Loss: 0.3383
Epoch [3/10] I-JEPA Loss: 0.3873
Epoch [4/10] I-JEPA Loss: 0.4656
Epoch [5/10] I-JEPA Loss: 0.3674
Epoch [6/10] I-JEPA Loss: 0.2957
Epoch [7/10] I-JEPA Loss: 0.2678
Epoch [8/10] I-JEPA Loss: 0.2457
Epoch [9/10] I-JEPA Loss: 0.2291
Epoch [10/10] I-JEPA Loss: 0.2169
Linear Probe Epoch [1/10] Loss: 1.7996 Train Accuracy: 37.32% Test Accuracy: 41.49%
Linear Probe Epoch [2/10] Loss: 1.6298 Train Accuracy: 42.41% Test Accuracy: 43.06%
Linear Probe Epoch [3/10] Loss: 1.5892 Train Accuracy: 43.86% Test Accuracy: 44.17%
Linear Probe Epoch [4/10] Loss: 1.5673 Train Accuracy: 44.32% Test Accuracy: 45.08%
Linear Probe Epoch [5/10] Loss: 1.5506 Train Accuracy: 44.86% Test Accuracy: 45.26%
Linear Probe Epoch [6/10] Loss: 1.5401 Train Accuracy: 45.16% Test Accuracy: 45.84%
Linear Probe Epoch [7/10] Loss: 1.5302 Train Accuracy: 45.51% Test Accuracy: 45.83%
Linear Probe Epoch [8/10] Loss: 1.5222 Train Accuracy: 46.04% Test Accuracy: 46.4